# ch04 Mini Experiment — Deep Neural Networks
**Prince, Understanding Deep Learning, Ch. 4**

This notebook trains a 3-layer deep ReLU network from scratch (NumPy only) to demonstrate:
- Forward pass through multiple hidden layers
- Manual backpropagation via the chain rule
- Depth enabling non-linear composition

---

## Theory: Forward Pass

A deep network with $L$ hidden layers computes:
$$\mathbf{h}^{(l)} = \phi\!\left(W^{(l)}\mathbf{h}^{(l-1)} + \mathbf{b}^{(l)}\right), \quad l = 1, \ldots, L$$
$$\hat{\mathbf{y}} = W^{(L+1)}\mathbf{h}^{(L)} + \mathbf{b}^{(L+1)}$$

**Why $\phi$ is mandatory:** without it, $W^{(2)}(W^{(1)}\mathbf{x}+\mathbf{b}^{(1)})+\mathbf{b}^{(2)} = \tilde{W}\mathbf{x}+\tilde{\mathbf{b}}$ — any depth collapses to one affine map.

**Architecture this notebook:** $2 \to 8 \to 8 \to 1$, ReLU hidden layers, linear output.
**Task:** $f(x_1, x_2) = \sin(x_1)\cos(x_2)$ — requires non-linear composition across layers.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(42)

def make_data(n=300):
    X = RNG.uniform(-np.pi, np.pi, (n, 2))
    y = (np.sin(X[:, 0]) * np.cos(X[:, 1])).reshape(-1, 1)
    return X, y

X, y = make_data()
print(f"X shape: {X.shape}, y shape: {y.shape}, y range: [{y.min():.2f}, {y.max():.2f}]")


## Theory: He Initialisation

For ReLU layers, **He (Kaiming) init** sets $\sigma^2 = 2/n_{in}$:
$$W^{(l)} \sim \mathcal{N}\!\left(0,\; \frac{2}{D_{l-1}}\right)$$

**Why 2/n:** ReLU zeros ~50% of outputs. Without the factor of 2, variance halves at each layer — activations shrink to zero, gradients vanish before reaching layer 1.


In [ ]:
def relu(z): return np.maximum(0.0, z)
def relu_grad(z): return (z > 0).astype(float)

def init_params(dims):
    params = []
    for i in range(len(dims) - 1):
        fan_in = dims[i]
        W = RNG.standard_normal((dims[i], dims[i+1])) * np.sqrt(2.0 / fan_in)
        b = np.zeros((1, dims[i+1]))
        params.append({"W": W, "b": b})
    return params

params = init_params([2, 8, 8, 1])
print("Param shapes:", [(p["W"].shape, p["b"].shape) for p in params])


## Theory: Backpropagation

Backprop is the chain rule applied in reverse through the computation graph.

For each layer $l$ (from $L$ down to 1):
$$\delta^{(l)} = \left[(W^{(l+1)})^\top \delta^{(l+1)}\right] \odot \phi'(\mathbf{z}^{(l)})$$
$$\frac{\partial \mathcal{L}}{\partial W^{(l)}} = \delta^{(l)} (\mathbf{h}^{(l-1)})^\top, \qquad \frac{\partial \mathcal{L}}{\partial \mathbf{b}^{(l)}} = \delta^{(l)}$$

**ReLU gradient in backprop:** $\phi'(\mathbf{z}^{(l)}) = \mathbf{1}[\mathbf{z}^{(l)} > 0]$ — binary gate, never a shrinking fraction.  
**Cache requirement:** the forward pass must store $\mathbf{z}^{(l)}$ and $\mathbf{h}^{(l)}$ at every layer — backprop reuses them.


In [ ]:
def forward(X, params):
    cache = []
    h = X
    for i, p in enumerate(params):
        z = h @ p["W"] + p["b"]
        is_last = (i == len(params) - 1)
        h_next = z if is_last else relu(z)
        cache.append({"h_in": h, "z": z, "h_out": h_next})
        h = h_next
    return h, cache

def backward(y_hat, y, params, cache):
    n = y.shape[0]
    grads = [{"W": np.zeros_like(p["W"]), "b": np.zeros_like(p["b"])} for p in params]
    delta = (2.0 / n) * (y_hat - y)
    for l in range(len(params) - 1, -1, -1):
        grads[l]["W"] = cache[l]["h_in"].T @ delta
        grads[l]["b"] = delta.sum(axis=0, keepdims=True)
        if l > 0:
            delta = (delta @ params[l]["W"].T) * relu_grad(cache[l-1]["z"])
    return grads

def update(params, grads, lr=0.01):
    return [{"W": p["W"] - lr * g["W"], "b": p["b"] - lr * g["b"]}
            for p, g in zip(params, grads)]

def mse(y_hat, y): return float(np.mean((y_hat - y) ** 2))

# Verify shapes
y_hat, cache = forward(X, params)
grads = backward(y_hat, y, params, cache)
print(f"Output shape: {y_hat.shape}, Loss: {mse(y_hat, y):.4f}")
print(f"Grad shapes OK: {all(g['W'].shape == p['W'].shape for g, p in zip(grads, params))}")


## Training Loop

200 steps of vanilla gradient descent.  
We expect the loss to halve — demonstrating that depth + backprop learns a non-trivial compositional function.


In [ ]:
params = init_params([2, 8, 8, 1])
y_hat_init, _ = forward(X, params)
loss_init = mse(y_hat_init, y)
losses = [loss_init]

for step in range(200):
    y_hat, cache = forward(X, params)
    grads = backward(y_hat, y, params, cache)
    params = update(params, grads, lr=0.01)
    if step % 20 == 0:
        losses.append(mse(y_hat, y))

y_hat_final, _ = forward(X, params)
loss_final = mse(y_hat_final, y)
print(f"Loss: {loss_init:.4f} → {loss_final:.4f}  (reduction: {loss_init/loss_final:.1f}x)")
assert loss_final < loss_init * 0.5, "Loss did not halve!"
print("Assertion passed: loss < 50% of initial")


## Visualisation: Loss Curve

A clean loss curve confirms gradient flow is working through all layers.  
Spikes indicate the learning rate could be tuned; monotone decrease confirms stable training.


In [ ]:
plt.style.use("dark_background")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(losses, color="#66d9e8", linewidth=2)
axes[0].set_xlabel("Step (×20)", color="white")
axes[0].set_ylabel("MSE Loss", color="white")
axes[0].set_title("Training Loss — 3-Layer ReLU Net", color="white")
axes[0].tick_params(colors="white")

# Predictions vs truth (first input dimension slice)
idx = np.argsort(X[:, 0])
axes[1].scatter(X[idx, 0], y[idx], s=8, color="#a9dc76", alpha=0.5, label="True")
axes[1].scatter(X[idx, 0], y_hat_final[idx], s=8, color="#ff6188", alpha=0.5, label="Predicted")
axes[1].set_xlabel("x₁", color="white")
axes[1].set_ylabel("f(x₁, x₂)", color="white")
axes[1].set_title("True vs Predicted", color="white")
axes[1].legend(facecolor="#1e1e1e", labelcolor="white")
axes[1].tick_params(colors="white")

plt.tight_layout()
plt.savefig("ch04_loss_curve.png", dpi=120, bbox_inches="tight", facecolor="#1e1e1e")
plt.show()
print("Plot saved.")
